In [ ]:
!pip install -q \
transformers \
accelerate \
bitsandbytes \
sentencepiece \
huggingface_hub \
scikit-learn \
pandas \
seaborn \
tqdm

In [ ]:
!pip install -q einops

In [ ]:
# Core
import os
import json
import pickle
import numpy as np
import pandas as pd

# PyTorch
import torch

# HuggingFace
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# ML metrics
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    brier_score_loss
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from tqdm import tqdm

In [ ]:
# ================================================================
# DVERSARIAL HUMANIZATION
# Objective: How far can AI text be modified before neural
#            detectors can no longer reliably detect it?
# Levels: L0 (original AI) → L1 (light humanization) → L2 (heavy humanization)
# ================================================================

# ================================================================
# CELL: IMPORTS & SETUP
# ================================================================

import pandas as pd
import numpy as np
import torch
import json
import os
import pickle
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForCausalLM, BitsAndBytesConfig
)
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

os.makedirs("./stage4_results", exist_ok=True)
SEED = 42
N_SAMPLES = 200   # per dataset — L4 safe

print("✅ Setup complete")


# ================================================================
# CELL: CONFIG
# ================================================================

HF_USER  = "Moodlerz"
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

# Detectors to evaluate — load from HF Hub
DETECTORS = {
    "BERT-HC3":       f"{HF_USER}/bert-detector-hc3",
    "RoBERTa-HC3":    f"{HF_USER}/roberta-detector-hc3",
    "ELECTRA-HC3":    f"{HF_USER}/electra-detector-hc3",
    "DistilBERT-HC3": f"{HF_USER}/distilbert-detector-hc3",
    "DeBERTa-HC3":    f"{HF_USER}/deberta-v3-detector-hc3",
}

# Humanization LLM — 4-bit quantized to fit on L4
HUMANIZER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Humanization prompts — increasing intensity
PROMPTS = {
    "L1": (
        "Rewrite the following AI-generated text to sound more natural and human. "
        "Use varied sentence lengths, occasional informal language, and avoid overly "
        "structured or formulaic writing patterns. Do not add new information or change "
        "the meaning. Return ONLY the rewritten text with no preamble or explanation.\n\n"
        "Text:\n{text}\n\nRewritten:"
    ),
    "L2": (
        "Rewrite the following text so it reads exactly like something a real person "
        "typed casually. Use contractions, natural imperfections, conversational tone, "
        "and deliberately avoid AI-like patterns such as numbered lists, formal transitions "
        "like 'Furthermore' or 'In conclusion', and overly complete sentences. You may "
        "break grammar rules slightly if it sounds more human. Do not add new information. "
        "Return ONLY the rewritten text with no preamble or explanation.\n\n"
        "Text:\n{text}\n\nRewritten:"
    ),
}

print("✅ Config set")
print(f"   Detectors   : {list(DETECTORS.keys())}")
print(f"   Humanizer   : {HUMANIZER_MODEL}")
print(f"   N samples   : {N_SAMPLES} per dataset")
print(f"   Levels      : L0 → L1 → L2")


# ================================================================
# CELL: LOAD & SAMPLE DATA
# ================================================================

hc3_test  = pd.read_csv("hc3_test.csv")
eli5_test = pd.read_csv("eli5_test.csv")

def encode_labels(labels):
    return (labels == 'llm').astype(int)

hc3_test['label_encoded']  = encode_labels(hc3_test['label'])
eli5_test['label_encoded'] = encode_labels(eli5_test['label'])

rng = np.random.default_rng(SEED)

# Sample N_SAMPLES LLM-generated texts from each dataset
hc3_llm  = hc3_test[hc3_test['label'] == 'llm'].reset_index(drop=True)
eli5_llm = eli5_test[eli5_test['label'] == 'llm'].reset_index(drop=True)

hc3_idx  = rng.choice(len(hc3_llm),  size=N_SAMPLES, replace=False)
eli5_idx = rng.choice(len(eli5_llm), size=N_SAMPLES, replace=False)

hc3_sample  = hc3_llm.iloc[hc3_idx].reset_index(drop=True)
eli5_sample = eli5_llm.iloc[eli5_idx].reset_index(drop=True)

# Store L0 texts
texts = {
    "hc3":  {"L0": hc3_sample['text'].tolist()},
    "eli5": {"L0": eli5_sample['text'].tolist()},
}

print(f"✅ Data sampled:")
print(f"   HC3  LLM samples : {len(hc3_sample)}")
print(f"   ELI5 LLM samples : {len(eli5_sample)}")
print(f"   Sample length stats (HC3)  — mean: {hc3_sample['text'].str.len().mean():.0f} chars")
print(f"   Sample length stats (ELI5) — mean: {eli5_sample['text'].str.len().mean():.0f} chars")


# ================================================================
# CELL: LOAD HUMANIZATION LLM (4-BIT QUANTIZED)
# ================================================================

print(f"\n⏳ Loading {HUMANIZER_MODEL} in 4-bit ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

humanizer_tok = AutoTokenizer.from_pretrained(
    HUMANIZER_MODEL, token=HF_TOKEN)
humanizer_tok.pad_token = humanizer_tok.eos_token
humanizer_tok.padding_side = "left"

humanizer_model = AutoModelForCausalLM.from_pretrained(
    HUMANIZER_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)
humanizer_model.eval()

print(f"✅ Humanizer loaded")
mem = torch.cuda.memory_allocated() / 1e9
print(f"   VRAM used: {mem:.2f} GB")


# ================================================================
# CELL: HUMANIZATION FUNCTION
# ================================================================

def humanize_texts(texts_list, level, batch_size=4, max_new_tokens=256):
    """
    Humanize a list of texts at a given level (L1 or L2).
    Uses the prompt template for that level.
    Returns list of humanized texts, same length as input.
    """
    prompt_template = PROMPTS[level]
    humanized = []

    for i in tqdm(range(0, len(texts_list), batch_size),
                  desc=f"Humanizing ({level})"):
        batch = texts_list[i:i+batch_size]

        # Build chat-style prompts
        prompts = []
        for text in batch:
            # Truncate very long texts to avoid OOM
            truncated = text[:800] if len(text) > 800 else text
            filled    = prompt_template.format(text=truncated)
            prompts.append(filled)

        enc = humanizer_tok(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            out = humanizer_model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.8,
                top_p=0.92,
                pad_token_id=humanizer_tok.eos_token_id,
                repetition_penalty=1.1
            )

        # Decode only newly generated tokens
        input_len = enc["input_ids"].shape[1]
        for j, seq in enumerate(out):
            new_tokens = seq[input_len:]
            decoded    = humanizer_tok.decode(
                            new_tokens, skip_special_tokens=True).strip()

            # Fallback: if empty output, keep original
            if len(decoded) < 20:
                decoded = batch[j]

            humanized.append(decoded)

    return humanized


# ================================================================
# CELL: RUN L1 AND L2 HUMANIZATION
# ================================================================

for dataset_name in ["hc3", "eli5"]:
    print(f"\n{'='*60}")
    print(f"Humanizing {dataset_name.upper()} texts")
    print(f"{'='*60}")

    l0_texts = texts[dataset_name]["L0"]

    # L1 humanization from L0
    print(f"\n▶ Level 1 humanization ...")
    l1_texts = humanize_texts(l0_texts, level="L1", batch_size=4)
    texts[dataset_name]["L1"] = l1_texts
    print(f"   ✅ L1 complete — {len(l1_texts)} texts")

    # L2 humanization from L1 output (iterative)
    print(f"\n▶ Level 2 humanization (from L1 output) ...")
    l2_texts = humanize_texts(l1_texts, level="L2", batch_size=4)
    texts[dataset_name]["L2"] = l2_texts
    print(f"   ✅ L2 complete — {len(l2_texts)} texts")

# Save humanized texts
with open("./stage4_results/humanized_texts.pkl", "wb") as f:
    pickle.dump(texts, f)
print("\n✅ Humanized texts saved")

# Free humanizer VRAM before loading detectors
del humanizer_model, humanizer_tok
torch.cuda.empty_cache()
print(f"✅ Humanizer freed — VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")


# ================================================================
# CELL: DETECTION FUNCTION
# ================================================================

def detect_texts(texts_list, detector_name, repo_id, batch_size=32):
    """
    Run a detector on a list of texts.
    Returns array of P(LLM) scores.
    """
    tok = AutoTokenizer.from_pretrained(repo_id, token=HF_TOKEN)
    mdl = AutoModelForSequenceClassification.from_pretrained(
            repo_id, token=HF_TOKEN)
    mdl.to(device)
    mdl.eval()

    all_scores = []
    for i in range(0, len(texts_list), batch_size):
        batch = texts_list[i:i+batch_size]
        enc   = tok(batch, return_tensors="pt", padding=True,
                    truncation=True, max_length=512).to(device)
        with torch.no_grad():
            logits = mdl(**enc).logits
        probs  = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_scores.extend(probs)

    del mdl, tok
    torch.cuda.empty_cache()
    return np.array(all_scores)


# ================================================================
# CELL: RUN DETECTION ACROSS ALL LEVELS
# ================================================================

hc3_human  = hc3_test[hc3_test['label'] == 'human'].sample(
                N_SAMPLES, random_state=SEED)['text'].tolist()
eli5_human = eli5_test[eli5_test['label'] == 'human'].sample(
                N_SAMPLES, random_state=SEED)['text'].tolist()

human_texts = {"hc3": hc3_human, "eli5": eli5_human}

# y_true: 1 = AI/humanized (still being tested), 0 = human
# We use the same human texts as the negative class at every level
# The question is: does the detector still think the humanized text is AI?
y_true_base = np.array([1]*N_SAMPLES + [0]*N_SAMPLES)

results_4 = {}   # results_4[detector][dataset][level] = {scores, metrics}

LEVELS = ["L0", "L1", "L2"]

for det_name, repo_id in DETECTORS.items():
    print(f"\n{'='*60}")
    print(f"Detector: {det_name}")
    print(f"{'='*60}")
    results_4[det_name] = {}

    for dataset_name in ["hc3", "eli5"]:
        results_4[det_name][dataset_name] = {}

        for level in LEVELS:
            print(f"  ▶ {dataset_name.upper()} | {level} ...", end=" ")

            ai_texts  = texts[dataset_name][level]
            neg_texts = human_texts[dataset_name]
            all_texts = ai_texts + neg_texts

            scores = detect_texts(all_texts, det_name, repo_id)
            ai_scores    = scores[:N_SAMPLES]
            human_scores = scores[N_SAMPLES:]

            auroc    = roc_auc_score(y_true_base, scores)
            brier    = brier_score_loss(y_true_base, scores)
            acc      = accuracy_score(y_true_base, (scores > 0.5).astype(int))
            det_rate = (ai_scores > 0.5).mean()   # % of AI texts still detected

            results_4[det_name][dataset_name][level] = {
                "scores":      scores,
                "ai_scores":   ai_scores,
                "human_scores":human_scores,
                "auroc":       auroc,
                "brier":       brier,
                "accuracy":    acc,
                "detection_rate": det_rate,
                "mean_ai_score":  ai_scores.mean(),
                "mean_human_score": human_scores.mean(),
            }
            print(f"AUROC={auroc:.4f}  DetRate={det_rate:.3f}")

# Save results
with open("./stage4_results/results_4.pkl", "wb") as f:
    pickle.dump(results_4, f)
print("\n✅ Detection results saved")


# ================================================================
# CELL: SUMMARY TABLE
# ================================================================

rows = []
for det_name in results_4:
    for dataset_name in ["hc3", "eli5"]:
        for level in LEVELS:
            r = results_4[det_name][dataset_name][level]
            rows.append({
                "Detector":        det_name,
                "Dataset":         dataset_name,
                "Level":           level,
                "AUROC":           r["auroc"],
                "Detection_Rate":  r["detection_rate"],
                "Mean_AI_Score":   r["mean_ai_score"],
                "Mean_Human_Score":r["mean_human_score"],
                "Brier":           r["brier"],
            })

summary_df = pd.DataFrame(rows)
summary_df.to_csv("./stage4_results/summary_4.csv", index=False)

print("\n" + "="*80)
print("STAGE 4 SUMMARY")
print("="*80)
print(summary_df.round(4).to_string(index=False))


# ================================================================
# CELL: AUROC DEGRADATION PLOT
# ================================================================

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
fig.suptitle("Stage 4: AUROC Across Humanization Levels\n"
             "L0=Original AI  L1=Light Humanization  L2=Heavy Humanization",
             fontsize=14, fontweight="bold")

det_names = list(results_4.keys())
colors    = {"hc3": "#3498db", "eli5": "#e74c3c"}

for col, det_name in enumerate(det_names):
    for row, dataset_name in enumerate(["hc3", "eli5"]):
        ax     = axes[row][col]
        aurocs = [results_4[det_name][dataset_name][lv]["auroc"]
                  for lv in LEVELS]
        det_rates = [results_4[det_name][dataset_name][lv]["detection_rate"]
                     for lv in LEVELS]

        ax.plot(LEVELS, aurocs, "o-", linewidth=2.5,
                color=colors[dataset_name], label="AUROC", markersize=8)
        ax.plot(LEVELS, det_rates, "s--", linewidth=2,
                color=colors[dataset_name], alpha=0.5,
                label="Detection Rate", markersize=7)

        ax.axhline(0.5, color="grey", linestyle=":", alpha=0.5,
                   label="Random baseline")
        ax.set_ylim(0, 1.05)
        ax.set_title(f"{det_name}\n({dataset_name.upper()})",
                     fontsize=9, fontweight="bold")
        ax.set_ylabel("Score", fontsize=8)
        ax.legend(fontsize=6)
        ax.grid(alpha=0.3)

        # Annotate AUROC values
        for i, (lv, auc) in enumerate(zip(LEVELS, aurocs)):
            ax.annotate(f"{auc:.3f}", (i, auc),
                        textcoords="offset points",
                        xytext=(0, 8), fontsize=7, ha="center")

plt.tight_layout()
plt.savefig("./stage4_results/auroc_degradation.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("📊 Saved: auroc_degradation.png")


# ================================================================
# CELL: DETECTION RATE HEATMAP
# ================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Stage 4: Detection Rate (% AI texts scored > 0.5)\n"
             "Rows = Detectors | Cols = Source LLM Level",
             fontsize=13, fontweight="bold")

for ax, dataset_name in zip(axes[:2], ["hc3", "eli5"]):
    pivot_data = {}
    for det_name in results_4:
        pivot_data[det_name] = {
            lv: results_4[det_name][dataset_name][lv]["detection_rate"]
            for lv in LEVELS
        }
    pivot = pd.DataFrame(pivot_data).T
    pivot.columns = LEVELS

    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn_r",
                vmin=0, vmax=1, ax=ax, linewidths=0.5,
                cbar_kws={"label": "Detection Rate"})
    ax.set_title(f"Detection Rate — {dataset_name.upper()}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Humanization Level")
    ax.set_ylabel("Detector")

# AUROC drop heatmap (L0 → L2 drop)
ax = axes[2]
drop_data = {}
for det_name in results_4:
    drop_data[det_name] = {
        ds: (results_4[det_name][ds]["L0"]["auroc"] -
             results_4[det_name][ds]["L2"]["auroc"])
        for ds in ["hc3", "eli5"]
    }
drop_df = pd.DataFrame(drop_data).T
drop_df.columns = ["HC3 Drop", "ELI5 Drop"]

sns.heatmap(drop_df, annot=True, fmt=".3f", cmap="Reds",
            vmin=0, ax=ax, linewidths=0.5,
            cbar_kws={"label": "AUROC Drop L0→L2"})
ax.set_title("AUROC Drop: L0 → L2\n(higher = more evaded)",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Detector")

plt.tight_layout()
plt.savefig("./stage4_results/detection_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("📊 Saved: detection_heatmap.png")


# ================================================================
# CELL: SCORE DISTRIBUTION SHIFT PLOT
# ================================================================

fig, axes = plt.subplots(len(det_names), 3, figsize=(18, 4*len(det_names)))
fig.suptitle("Stage 4: P(AI) Score Distribution Shift Across Humanization Levels\n"
             "Blue=Human  Red=AI/Humanized",
             fontsize=13, fontweight="bold")

for row, det_name in enumerate(det_names):
    for col, level in enumerate(LEVELS):
        ax = axes[row][col]

        # Use HC3 for distribution plots
        ai_scores    = results_4[det_name]["hc3"][level]["ai_scores"]
        human_scores = results_4[det_name]["hc3"][level]["human_scores"]

        ax.hist(human_scores, bins=30, alpha=0.6, color="#3498db",
                label="Human", range=(0,1), density=True)
        ax.hist(ai_scores,    bins=30, alpha=0.6, color="#e74c3c",
                label=f"AI ({level})", range=(0,1), density=True)
        ax.axvline(0.5, color="black", linestyle="--", alpha=0.4)

        auc = results_4[det_name]["hc3"][level]["auroc"]
        dr  = results_4[det_name]["hc3"][level]["detection_rate"]
        ax.set_title(f"{det_name} | {level}\nAUROC={auc:.3f}  DetRate={dr:.3f}",
                     fontsize=9, fontweight="bold")
        ax.set_xlim(0, 1)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("./stage4_results/score_distributions_4.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("📊 Saved: score_distributions_4.png")

print("\n" + "="*60)
print("✅ STAGE 4 COMPLETE")
print("="*60)
print(f"📊 Results in: ./stage4_results/")

In [ ]:
# ================================================================
# SANITY CHECK & DATASET EXPORT
# ================================================================

import pandas as pd
import numpy as np
import pickle
import os
import json
from IPython.display import display

# ================================================================
# CELL S-1: LOAD HUMANIZED TEXTS
# ================================================================

with open("./stage4_results/humanized_texts.pkl", "rb") as f:
    texts = pickle.load(f)

with open("./stage4_results/results_4.pkl", "rb") as f:
    results_4 = pickle.load(f)

print("✅ Loaded humanized_texts.pkl and results_4.pkl")
print(f"   Datasets : {list(texts.keys())}")
print(f"   Levels   : {list(texts['hc3'].keys())}")
print(f"   Samples  : {len(texts['hc3']['L0'])} per dataset per level")


# ================================================================
# CELL S-2: SIDE-BY-SIDE TEXT SAMPLES
# ================================================================

N_SHOW = 3  # number of samples to display per dataset

print("\n" + "="*80)
print("SAMPLE TEXTS — SIDE BY SIDE ACROSS LEVELS")
print("="*80)

for dataset_name in ["hc3", "eli5"]:
    print(f"\n{'#'*80}")
    print(f"DATASET: {dataset_name.upper()}")
    print(f"{'#'*80}")

    for idx in range(N_SHOW):
        print(f"\n{'─'*80}")
        print(f"Sample #{idx+1}")
        print(f"{'─'*80}")

        for level in ["L0", "L1", "L2"]:
            text = texts[dataset_name][level][idx]
            # Truncate for display
            display_text = text[:400] + " ..." if len(text) > 400 else text

            print(f"\n  [{level}] ({len(text)} chars)")
            print(f"  {display_text}")

        print()


# ================================================================
# CELL S-3: QUANTITATIVE SANITY CHECKS
# ================================================================

print("\n" + "="*80)
print("QUANTITATIVE SANITY CHECKS")
print("="*80)

sanity_rows = []

for dataset_name in ["hc3", "eli5"]:
    for level in ["L0", "L1", "L2"]:
        sample_texts = texts[dataset_name][level]

        lengths      = [len(t) for t in sample_texts]
        word_counts  = [len(t.split()) for t in sample_texts]
        empty_count  = sum(1 for t in sample_texts if len(t.strip()) < 20)
        # Check for texts that look like the prompt leaked into output
        prompt_leak  = sum(1 for t in sample_texts
                          if "Rewrite" in t[:50] or "Text:" in t[:50]
                          or "Rewritten:" in t[:50])
        # Check for texts suspiciously identical to L0
        if level != "L0":
            l0_texts   = texts[dataset_name]["L0"]
            unchanged  = sum(1 for a, b in zip(l0_texts, sample_texts)
                            if a.strip() == b.strip())
        else:
            unchanged  = 0

        sanity_rows.append({
            "Dataset":        dataset_name,
            "Level":          level,
            "Mean_Chars":     np.mean(lengths),
            "Std_Chars":      np.std(lengths),
            "Min_Chars":      np.min(lengths),
            "Max_Chars":      np.max(lengths),
            "Mean_Words":     np.mean(word_counts),
            "Empty_Count":    empty_count,
            "Prompt_Leaks":   prompt_leak,
            "Unchanged_vs_L0":unchanged,
        })

sanity_df = pd.DataFrame(sanity_rows)
print(sanity_df.round(1).to_string(index=False))

# Flag issues
print("\n" + "="*80)
print("SANITY FLAGS")
print("="*80)
issues_found = False
for _, row in sanity_df.iterrows():
    flags = []
    if row["Empty_Count"] > 0:
        flags.append(f"⚠️  {int(row['Empty_Count'])} empty/very short outputs")
    if row["Prompt_Leaks"] > 0:
        flags.append(f"⚠️  {int(row['Prompt_Leaks'])} prompt leaks detected")
    if row["Unchanged_vs_L0"] > 0:
        flags.append(f"⚠️  {int(row['Unchanged_vs_L0'])} texts unchanged from L0")
    if row["Mean_Chars"] < 100:
        flags.append(f"⚠️  Very short mean length ({row['Mean_Chars']:.0f} chars)")

    if flags:
        issues_found = True
        print(f"\n  {row['Dataset'].upper()} | {row['Level']}:")
        for flag in flags:
            print(f"    {flag}")

if not issues_found:
    print("  ✅ No issues detected across all datasets and levels")


# ================================================================
# CELL S-4: SCORE DISTRIBUTION SANITY CHECK
# ================================================================

print("\n" + "="*80)
print("DETECTION SCORE SANITY CHECK")
print("="*80)

score_rows = []
for det_name in results_4:
    for dataset_name in ["hc3", "eli5"]:
        for level in ["L0", "L1", "L2"]:
            r = results_4[det_name][dataset_name][level]
            ai_s = r["ai_scores"]
            score_rows.append({
                "Detector":   det_name,
                "Dataset":    dataset_name,
                "Level":      level,
                "Mean":       ai_s.mean(),
                "Std":        ai_s.std(),
                "Min":        ai_s.min(),
                "Max":        ai_s.max(),
                "Pct_above_0.5": (ai_s > 0.5).mean(),
                "Pct_below_0.5": (ai_s < 0.5).mean(),
            })

score_df = pd.DataFrame(score_rows)
print(score_df.round(4).to_string(index=False))

# Flag collapsed detectors at any level
print("\n" + "="*80)
print("SCORE FLAGS")
print("="*80)
score_issues = False
for _, row in score_df.iterrows():
    if row["Std"] < 0.05:
        print(f"  ⚠️  {row['Detector']} | {row['Dataset']} | {row['Level']}"
              f" — LOW STD ({row['Std']:.4f}) — possible collapse")
        score_issues = True
if not score_issues:
    print("  ✅ All score distributions look healthy")


# ================================================================
# CELL S-5: BUILD & EXPORT FULL DATASET
# ================================================================

print("\n" + "="*80)
print("BUILDING EXPORT DATASET")
print("="*80)

export_rows = []

for dataset_name in ["hc3", "eli5"]:
    for level in ["L0", "L1", "L2"]:
        sample_texts = texts[dataset_name][level]

        for idx, text in enumerate(sample_texts):
            row = {
                "sample_id":   f"{dataset_name}_{idx:03d}",
                "dataset":     dataset_name,
                "level":       level,
                "text":        text,
                "char_length": len(text),
                "word_count":  len(text.split()),
            }

            # Attach detection scores from every detector
            for det_name in results_4:
                ai_scores = results_4[det_name][dataset_name][level]["ai_scores"]
                if idx < len(ai_scores):
                    col_name      = det_name.lower().replace("-", "_") + "_score"
                    row[col_name] = round(float(ai_scores[idx]), 6)

            export_rows.append(row)

export_df = pd.DataFrame(export_rows)

print(f"✅ Export dataframe built")
print(f"   Rows    : {len(export_df)}")
print(f"   Columns : {list(export_df.columns)}")
print(f"\nPreview:")
print(export_df.head(6).to_string(index=False))

# Save as CSV
export_path_csv = "./stage4_results/humanized_dataset.csv"
export_df.to_csv(export_path_csv, index=False)
print(f"\n✅ Saved: {export_path_csv}")

# Save as JSON (preserves full text without CSV truncation issues)
export_path_json = "./stage4_results/humanized_dataset.json"
export_df.to_json(export_path_json, orient="records", indent=2)
print(f"✅ Saved: {export_path_json}")

# Save metadata separately
metadata = {
    "n_samples_per_level":  len(texts["hc3"]["L0"]),
    "datasets":             ["hc3", "eli5"],
    "levels":               ["L0", "L1", "L2"],
    "detectors":            list(results_4.keys()),
    "humanizer_model":      "Qwen/Qwen2.5-1.5B-Instruct",
    "level_descriptions": {
        "L0": "Original AI-generated text — no modification",
        "L1": "Light humanization — varied sentence length, informal tone",
        "L2": "Heavy humanization — applied to L1 output, aggressive style shift"
    },
    "score_column_meaning": "P(LLM) — probability the text is AI-generated (0=human, 1=AI)"
}

with open("./stage4_results/dataset_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Saved: dataset_metadata.json")


# ================================================================
# CELL S-6: DOWNLOAD ALL FILES
# ================================================================

from google.colab import files
import zipfile

# Zip everything together
zip_path = "./stage4_results/stage4_complete.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("./stage4_results/humanized_dataset.csv",  "humanized_dataset.csv")
    zf.write("./stage4_results/humanized_dataset.json", "humanized_dataset.json")
    zf.write("./stage4_results/dataset_metadata.json",  "dataset_metadata.json")
    zf.write("./stage4_results/summary_4.csv",          "summary_4.csv")
    zf.write("./stage4_results/humanized_texts.pkl",    "humanized_texts.pkl")
    zf.write("./stage4_results/results_4.pkl",          "results_4.pkl")

print(f"✅ Zipped all files → {zip_path}")

# Download
files.download(zip_path)
print("✅ Download triggered")